In [ ]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def scrape_gfg(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return {"url": url, "status": "failed", "reason": f"HTTP {response.status_code}", "text": None}

        soup = BeautifulSoup(response.text, "html.parser")

        # GFG problem statement usually lives in these selectors
        selectors = [
            "div.MainArticleContent_articleMainContentCss__b_1_R",
            "div.article--viewer_content",
            "div.content",
        ]

        for selector in selectors:
            el = soup.select_one(selector)
            if el:
                text = el.get_text(separator=" ", strip=True)
                if len(text) > 200:  # minimum length to be a real problem
                    return {"url": url, "status": "success", "text": text[:850]}

        return {"url": url, "status": "failed", "reason": "no matching selector", "text": None}

    except Exception as e:
        return {"url": url, "status": "error", "reason": str(e), "text": None}


# Test on a few links from your JSON
test_links = [
    "https://www.geeksforgeeks.org/activity-selection-problem-greedy-algo-1/",
    "https://www.geeksforgeeks.org/minimum-number-platforms-required-railwaybus-station/",
    "https://www.geeksforgeeks.org/floyd-warshall-algorithm-dp-16/",
    "https://www.geeksforgeeks.org/dynamic-programming-set-7-coin-change/",
    "https://www.geeksforgeeks.org/subset-sum-problem-dp-25/",
]

for url in test_links:
    result = scrape_gfg(url)
    print(f"\n{url}")
    print(f"Status: {result['status']}")
    if result['text']:
        print(f"Text preview: {result['text'][:850]}")
    else:
        print(f"Reason: {result.get('reason')}")
    time.sleep(1)  # be gentle

In [ ]:
def debug_gfg(url):
    response = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Print all div classes present on the page
    divs = soup.find_all("div", class_=True)
    classes = set()
    for div in divs:
        for c in div.get("class", []):
            classes.add(c)
    
    print(f"Status code: {response.status_code}")
    print(f"Page title: {soup.title.string if soup.title else 'None'}")
    print("\nAll div classes found:")
    for c in sorted(classes):
        print(f"  {c}")

debug_gfg("https://www.geeksforgeeks.org/activity-selection-problem-greedy-algo-1/")